# Exploratory Notebook: Temporal Feature Selection (exp_01)

**Goal:** Analyze monthly Sentinel-2 features using Jeffries-Matusita (JM) distance to identify months with strong genus discrimination.

**Outputs:**
- `outputs/phase_2/metadata/temporal_selection.json` (on Google Drive)
- Publication-quality plots saved to `outputs/phase_2/figures/exp_01_temporal`


In [ ]:
# ============================================================
# RUNTIME SETTINGS
# ============================================================
# Required: CPU (Standard)
# GPU: Not required
# High-RAM: Recommended for large datasets
#
# SETUP: Add GITHUB_TOKEN to Colab Secrets (key icon in sidebar)
# ============================================================

import subprocess
from google.colab import userdata

# Get GitHub token from Colab Secrets (for private repo access)
token = userdata.get("GITHUB_TOKEN")
if not token:
    raise ValueError(
        "GITHUB_TOKEN not found in Colab Secrets.\n"
        "1. Click the key icon in the left sidebar\n"
        "2. Add a secret named 'GITHUB_TOKEN' with your GitHub token\n"
        "3. Toggle 'Notebook access' ON"
    )

# Install package from private GitHub repo
repo_url = f"git+https://{token}@github.com/SilasPignotti/urban-tree-transfer.git"
subprocess.run(["pip", "install", repo_url, "-q"], check=True)

print("OK: Package installed")


In [ ]:
# Mount Google Drive for data files
from google.colab import drive
drive.mount("/content/drive")

print("Google Drive mounted")


In [ ]:
# Package imports
from urban_tree_transfer.config import PROJECT_CRS
from urban_tree_transfer.utils import ExecutionLog, save_figure, setup_plotting
from urban_tree_transfer.utils.plotting import PUBLICATION_STYLE

from itertools import combinations
from pathlib import Path
from datetime import datetime, timezone
import json
import re

import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

setup_plotting()
log = ExecutionLog("exp_01_temporal_analysis")

print("OK: Package imports complete")


In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

DRIVE_DIR = Path("/content/drive/MyDrive/dev/urban-tree-transfer")
INPUT_DIR = DRIVE_DIR / "data" / "phase_2_features"
OUTPUT_DIR = DRIVE_DIR / "outputs" / "phase_2"
METADATA_DIR = OUTPUT_DIR / "metadata"
LOGS_DIR = OUTPUT_DIR / "logs"
PLOTS_DIR = OUTPUT_DIR / "figures" / "exp_01_temporal"

CITIES = ["berlin", "leipzig"]
MONTHS = list(range(1, 13))

MIN_SAMPLES_PER_GENUS = 500
MAX_SAMPLES_PER_GENUS = 10000  # Balance: 10k gives stable JM estimates, ~80% faster than full dataset
RANDOM_SEED = 42

for d in [METADATA_DIR, LOGS_DIR, PLOTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Input:  {INPUT_DIR}")
print(f"Output: {METADATA_DIR}")
print(f"Plots:  {PLOTS_DIR}")
print(f"Cities: {CITIES}")


## Data Loading & Validation

Load Phase 2a feature data for Berlin and Leipzig and validate expected schema and temporal feature layout.

In [ ]:
log.start_step("Data Loading & Validation")

try:
    city_data = {}
    all_feature_cols = None
    feature_cols_by_month = None
    s2_feature_bases = None

    for city in CITIES:
        path = INPUT_DIR / f"trees_with_features_{city}.gpkg"
        print(f"Loading: {path}")
        df = gpd.read_file(path)

        # Basic validation
        required_cols = ["tree_id", "city", "genus_latin", "geometry"]
        missing = [c for c in required_cols if c not in df.columns]
        if missing:
            raise ValueError(f"Missing required columns: {missing}")

        # Identify S2 feature columns
        pattern = re.compile(r"^(.+)_([0-1][0-9])$")
        feature_cols = []
        feature_map = {}
        for col in df.columns:
            match = pattern.match(col)
            if match:
                base, month = match.group(1), int(match.group(2))
                if 1 <= month <= 12:
                    feature_cols.append(col)
                    feature_map.setdefault(month, []).append(col)

        if not feature_cols:
            raise ValueError("No Sentinel-2 feature columns found.")

        for month in MONTHS:
            if month not in feature_map:
                raise ValueError(f"Missing features for month {month:02d}")

        # Consistency checks across cities
        if all_feature_cols is None:
            all_feature_cols = sorted(feature_cols)
            feature_cols_by_month = {m: sorted(cols) for m, cols in feature_map.items()}
            s2_feature_bases = sorted({pattern.match(c).group(1) for c in all_feature_cols})
        else:
            if sorted(feature_cols) != all_feature_cols:
                raise ValueError(f"Feature columns mismatch between cities: {city}")

        # Filter to genera with sufficient samples
        genus_counts = df["genus_latin"].value_counts()
        viable_genera = genus_counts[genus_counts >= MIN_SAMPLES_PER_GENUS].index.tolist()
        df = df[df["genus_latin"].isin(viable_genera)].copy()

        # Optional sampling per genus to reduce compute time
        if MAX_SAMPLES_PER_GENUS is not None:
            sampled = []
            for genus in viable_genera:
                subset = df[df["genus_latin"] == genus]
                if len(subset) > MAX_SAMPLES_PER_GENUS:
                    subset = subset.sample(MAX_SAMPLES_PER_GENUS, random_state=RANDOM_SEED)
                sampled.append(subset)
            df = gpd.GeoDataFrame(pd.concat(sampled, ignore_index=True), crs=df.crs)

        city_data[city] = df
        print(f"  {city}: {len(df):,} trees, {len(viable_genera)} viable genera")

    log.end_step(status="success", records=sum(len(v) for v in city_data.values()))

except Exception as e:
    log.end_step(status="error", errors=[str(e)])
    raise


## JM Distance Computation

Compute Jeffries-Matusita distance per month, per city, and per genus pair. JM is computed per feature and averaged across all 23 Sentinel-2 features for each month.

In [ ]:
def compute_jm_distance(mu1: float, sigma1: float, mu2: float, sigma2: float) -> float:
    """Compute Jeffries-Matusita distance (standard formula)."""
    eps = 1e-6
    sigma1 = max(float(sigma1), eps)
    sigma2 = max(float(sigma2), eps)
    mean_diff = float(mu1) - float(mu2)

    sigma_avg = 0.5 * (sigma1 + sigma2)
    term1 = (1.0 / 8.0) * (mean_diff ** 2) / sigma_avg
    term2 = 0.5 * np.log(sigma_avg / np.sqrt(sigma1 * sigma2))
    b_distance = term1 + term2

    jm = 2.0 * (1.0 - np.exp(-b_distance))
    jm = float(np.clip(jm, 0.0, 2.0))
    return jm

def compute_feature_stats(df: pd.DataFrame, feature_cols: list[str]) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    grouped = df.groupby("genus_latin")[feature_cols]
    means = grouped.mean()
    stds = grouped.std(ddof=1)
    counts = grouped.count()
    return means, stds, counts

log.start_step("JM Distance Computation")

try:
    jm_records = []
    viable_genera_by_city = {}

    for city, df in city_data.items():
        genus_counts = df["genus_latin"].value_counts()
        viable_genera = genus_counts[genus_counts >= MIN_SAMPLES_PER_GENUS].index.tolist()
        viable_genera_by_city[city] = sorted(viable_genera)

        print(f"Computing stats for {city}...")
        means, stds, counts = compute_feature_stats(df, all_feature_cols)

        genus_pairs = list(combinations(viable_genera, 2))
        print(f"  Genus pairs: {len(genus_pairs)}")

        for month in MONTHS:
            month_cols = feature_cols_by_month[month]

            for genus_a, genus_b in genus_pairs:
                jm_values = []
                for col in month_cols:
                    if counts.loc[genus_a, col] < MIN_SAMPLES_PER_GENUS:
                        continue
                    if counts.loc[genus_b, col] < MIN_SAMPLES_PER_GENUS:
                        continue

                    mu1 = means.loc[genus_a, col]
                    mu2 = means.loc[genus_b, col]
                    sigma1 = stds.loc[genus_a, col]
                    sigma2 = stds.loc[genus_b, col]

                    if not np.isfinite(mu1) or not np.isfinite(mu2):
                        continue
                    if not np.isfinite(sigma1) or not np.isfinite(sigma2):
                        continue

                    jm = compute_jm_distance(mu1, sigma1, mu2, sigma2)
                    if np.isfinite(jm):
                        jm_values.append(jm)

                if jm_values:
                    jm_mean = float(np.mean(jm_values))
                    jm_records.append(
                        {
                            "city": city,
                            "month": month,
                            "genus_pair": f"{genus_a} vs {genus_b}",
                            "jm_distance": jm_mean,
                            "features_used": len(jm_values),
                        }
                    )

    jm_df = pd.DataFrame(jm_records)
    if jm_df.empty:
        raise ValueError("JM computation resulted in empty output. Check data and thresholds.")

    # Sanity checks
    if jm_df["jm_distance"].min() < 0 or jm_df["jm_distance"].max() > 2.0:
        raise ValueError("JM distances out of expected range [0, 2].")
    if jm_df["jm_distance"].isna().any():
        raise ValueError("NaN values detected in JM distances.")

    log.end_step(status="success", records=len(jm_df))

except Exception as e:
    log.end_step(status="error", errors=[str(e)])
    raise


## Visualization

Generate publication-quality plots for temporal JM trends and genus-pair heatmaps.

In [ ]:
# Plot 1: JM Distance Line Chart (Berlin vs Leipzig)
plot_df = (
    jm_df.groupby(["city", "month"])["jm_distance"]
    .agg(["mean", "std"])
    .reset_index()
)

city_colors = {
    "berlin": "#1f77b4",
    "leipzig": "#ff7f0e",
}

fig, ax = plt.subplots(figsize=(12, 6))
for city in CITIES:
    subset = plot_df[plot_df["city"] == city]
    ax.errorbar(
        subset["month"],
        subset["mean"],
        yerr=subset["std"],
        label=city.title(),
        color=city_colors.get(city),
        marker="o",
        linewidth=2,
        capsize=3,
    )

ax.set_title("Temporal Separability (JM Distance) by Month", fontsize=14)
ax.set_xlabel("Month", fontsize=12)
ax.set_ylabel("Mean JM Distance", fontsize=12)
ax.set_xticks(MONTHS)
ax.grid(True, alpha=0.3)
ax.legend(loc="best", fontsize=10)

save_figure(fig, PLOTS_DIR / "jm_distance_by_month.png", dpi=300)
print(f"Saved: {PLOTS_DIR / 'jm_distance_by_month.png'}")


In [ ]:
# Plot 2: JM Distance Heatmaps (top genus pairs)
TOP_GENUS_COUNT = 6  # 6 genera -> 15 pairs

for city in CITIES:
    df_city = city_data[city]
    top_genera = df_city["genus_latin"].value_counts().head(TOP_GENUS_COUNT).index.tolist()
    top_pairs = [f"{a} vs {b}" for a, b in combinations(top_genera, 2)]

    heatmap_df = jm_df[(jm_df["city"] == city) & (jm_df["genus_pair"].isin(top_pairs))]
    if heatmap_df.empty:
        print(f"No heatmap data for {city}.")
        continue

    pivot = heatmap_df.pivot(index="genus_pair", columns="month", values="jm_distance")
    pivot = pivot.reindex(index=top_pairs)

    fig, ax = plt.subplots(figsize=(14, 8))
    sns.heatmap(
        pivot,
        annot=True,
        fmt=".2f",
        cmap="viridis",
        cbar_kws={"label": "JM Distance"},
        ax=ax,
    )

    ax.set_title(
        f"Genus Separability Across Months ({city.title()})",
        fontsize=14,
    )
    ax.set_xlabel("Month", fontsize=12)
    ax.set_ylabel("Genus Pair", fontsize=12)
    ax.set_xticklabels([f"{m:02d}" for m in MONTHS], rotation=0)

    filename = f"jm_heatmap_{city}.png"
    save_figure(fig, PLOTS_DIR / filename, dpi=300)
    print(f"Saved: {PLOTS_DIR / filename}")


## Month Selection Logic

Select months using a top-N approach with seasonal balance to ensure phenological coverage.

In [ ]:
monthly_jm = jm_df.groupby("month")["jm_distance"].mean().sort_values(ascending=False)
threshold = float(monthly_jm.quantile(0.75))

def select_months_with_balance(monthly_series: pd.Series, top_n: int = 8) -> list[int]:
    seasons = {
        "spring": [3, 4, 5],
        "summer": [6, 7, 8],
        "autumn": [9, 10, 11],
        "winter": [12, 1, 2],
    }
    min_per_season = {
        "spring": 1,
        "summer": 2,
        "autumn": 1,
        "winter": 0,
    }

    selected = list(monthly_series.head(top_n).index)

    # Ensure seasonal balance
    for season, months in seasons.items():
        needed = min_per_season[season]
        current = [m for m in selected if m in months]
        if len(current) >= needed:
            continue
        candidates = [m for m in months if m not in selected]
        candidates = sorted(candidates, key=lambda m: monthly_series.get(m, -1), reverse=True)
        for m in candidates:
            if len(current) >= needed:
                break
            selected.append(m)
            current.append(m)

    # Cap size to 10 if seasonal additions push over
    if len(selected) > 10:
        selected = sorted(selected, key=lambda m: monthly_series.get(m, -1), reverse=True)[:10]

    selected = sorted(set(selected))
    return selected

selected_months = select_months_with_balance(monthly_jm, top_n=8)
rejected_months = [m for m in MONTHS if m not in selected_months]

print("Selected months:", selected_months)
print("Rejected months:", rejected_months)


## JSON Output

Save selected months and JM statistics for downstream use in Phase 2b.

In [ ]:
monthly_stats = (
    jm_df.groupby("month")["jm_distance"]
    .agg(["mean", "std", "min", "max"])
    .round(4)
)

viable_genera_all = sorted(set.intersection(*[set(v) for v in viable_genera_by_city.values()]))

output = {
    "analysis_date": datetime.now(timezone.utc).isoformat().replace("+00:00", "Z"),
    "input_files": [
        "trees_with_features_berlin.gpkg",
        "trees_with_features_leipzig.gpkg",
    ],
    "total_trees_analyzed": int(sum(len(df) for df in city_data.values())),
    "viable_genera": viable_genera_all,
    "genus_pairs_analyzed": int(jm_df["genus_pair"].nunique()),
    "monthly_jm_statistics": {
        str(month): {
            "mean": float(monthly_stats.loc[month, "mean"]),
            "std": float(monthly_stats.loc[month, "std"]),
            "min": float(monthly_stats.loc[month, "min"]),
            "max": float(monthly_stats.loc[month, "max"]),
        }
        for month in MONTHS
    },
    "selection_method": "top_n_with_seasonal_balance",
    "selection_threshold": float(threshold),
    "selected_months": selected_months,
    "rejected_months": rejected_months,
    "rationale": (
        "Selected 8 months with highest mean JM distance and seasonal coverage. "
        "Winter months show lower separability, so they are typically excluded unless needed for balance."
    ),
}

output_path = METADATA_DIR / "temporal_selection.json"
output_path.write_text(json.dumps(output, indent=2), encoding="utf-8")
print(f"Saved JSON: {output_path}")


## Summary & Manual Sync Instructions

In [ ]:
print("=" * 60)
print("TEMPORAL SELECTION SUMMARY")
print("=" * 60)
print("Total months analyzed: 12")
print(f"Months selected: {len(selected_months)}")
print(f"Selected: {selected_months}")
print(f"Mean JM (selected): {monthly_jm[selected_months].mean():.3f}")
selected_mean = monthly_jm[selected_months].mean()
if selected_mean < 0.7:
    interpretation = "low separability"
elif selected_mean < 1.0:
    interpretation = "moderate separability"
else:
    interpretation = "good separability"
print(f"Interpretation: {interpretation} (JM range: 0=overlap, 1=acceptable, 2=perfect)")
print(f"Mean JM (rejected): {monthly_jm[rejected_months].mean():.3f}")
print("\nSeasonal Coverage:")
print(f"  Spring (3-5):  {len([m for m in selected_months if 3 <= m <= 5])}/3")
print(f"  Summer (6-8):  {len([m for m in selected_months if 6 <= m <= 8])}/3")
print(f"  Autumn (9-11): {len([m for m in selected_months if 9 <= m <= 11])}/3")
print(f"  Winter (12,1,2): {len([m for m in selected_months if m in [12, 1, 2]])}/3")
print("\nJM Distance Quality:")
print(f"  Selected months mean: {monthly_jm[selected_months].mean():.3f}")
print(f"  Rejected months mean: {monthly_jm[rejected_months].mean():.3f}")
print(f"  Overall range: [{jm_df['jm_distance'].min():.3f}, {jm_df['jm_distance'].max():.3f}]")
print("\nNote: Legacy JM values were typically 0.5-1.2. Values are used for")
print("      relative comparison between months, not absolute thresholds.")

# Save execution log
log.summary()
log_path = LOGS_DIR / f"{log.notebook}_execution.json"
log.save(log_path)
print(f"Execution log saved: {log_path}")

print("\n--- JSON OUTPUT ---")
print(f"  {output_path.name}")

print("\n--- PLOTS CREATED ---")
for f in sorted(PLOTS_DIR.glob("*.png")):
    print(f"  {f.name}")


## ⚠️ Manual Sync Required

This notebook generated `temporal_selection.json` on Google Drive.

**Next Steps:**
1. Download `temporal_selection.json` from Drive to local repo
2. Place in: `outputs/phase_2/metadata/temporal_selection.json`
3. Commit to git
4. Push to GitHub
5. Continue with Task 6 (Quality Control implementation)

**File Path on Drive:**
`/content/drive/MyDrive/dev/urban-tree-transfer/outputs/phase_2/metadata/temporal_selection.json`
